# Notebook 27b — Wu 2003 Sequential Degradation Tracking for S-B3

This notebook mirrors notebook 27's sequential-tracking role, but under the reduced-scope `S-B3`
framing. Only endogenous reactor/column degradation is allowed to drift; `z_A0_eff` is fixed at
nominal and feed-fault scenarios are out of scope.


## Notebook contents

1. Load S-B3 posterior
2. Define 30-day reduced-scope degradation profile
3. Generate sequential observation windows
4. S-B3 sequential inference
5. Reduced-scope reference baseline
6. 30-day tracking figure
7. Metrics table

In [ ]:
import pathlib, pickle, sys, importlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT / 'src'))

import cstr_sbi.recycle.sb3 as sb3
sb3 = importlib.reload(sb3)
from cstr_sbi.recycle.sb3 import SB3_PARAM_NAMES, theta4_to_theta5, classify_fault_sb3
from cstr_sbi.recycle.simulator import nominal_warm_start, deterministic_window, noisy_replicates
from cstr_sbi.recycle.summaries import compute_summaries

SBI_LOGS = ROOT / 'sbi-logs'
RESULTS = ROOT / 'results'; RESULTS.mkdir(exist_ok=True)
FIGURES = ROOT / 'figures'; FIGURES.mkdir(exist_ok=True)

with open(SBI_LOGS / 'wu2003_posterior_sb3.pkl', 'rb') as f:
    payload = pickle.load(f)
posterior = payload['posterior']
y0_sb = nominal_warm_start('S-B')
print('loaded sb3 posterior:', payload.get('selected_seed'), payload.get('experiment'))

## 1. Load S-B3 Posterior

The tracking result is only meaningful if notebook 24b selected a calibrated posterior. This cell
therefore mirrors notebook 27's posterior-loading step, but points at the reduced-scope production
posterior.

In [ ]:
tracks = {
    'reactor_drift': np.stack([np.linspace(1.0, 0.65, 24), np.ones(24), np.ones(24), np.ones(24)], axis=1).astype(np.float32),
    'column_drift': np.stack([np.ones(24), np.ones(24), np.linspace(1.0, 0.80, 24), np.linspace(1.0, 0.70, 24)], axis=1).astype(np.float32),
}
rows = []
for track_name, theta4_path in tracks.items():
    for step, theta4 in enumerate(theta4_path):
        theta5 = theta4_to_theta5(theta4)
        from cstr_sbi.recycle.scenarios import RecycleScenarioConfig
        sc = RecycleScenarioConfig(id=1000 + step, name=f'{track_name}_{step:02d}', alpha=float(theta5[0]), beta_r=float(theta5[1]), eta_col=float(theta5[2]), xi_reb=float(theta5[3]), z_A0_eff=float(theta5[4]), mode='closed_loop', description='synthetic sb3 track')
        t_h, raw_obs = deterministic_window(sc, structure='S-B', y0=y0_sb)
        rng = np.random.default_rng(270001 + step)
        noisy = noisy_replicates(np.asarray(raw_obs), n_replicates=1, rng=rng, noise_pct=0.003)[0]
        s = compute_summaries(noisy, 'S-B', np.asarray(t_h))
        with torch.no_grad():
            samp = posterior.sample((400,), x=torch.tensor(s, dtype=torch.float32), show_progress_bars=False, reject_outside_prior=False).detach().cpu().numpy()
        pred = classify_fault_sb3(samp)
        means = samp.mean(axis=0)
        lo = np.percentile(samp, 5, axis=0)
        hi = np.percentile(samp, 95, axis=0)
        row = {'track': track_name, 'step': step, 'pred_class': pred['class']}
        for i, name in enumerate(SB3_PARAM_NAMES):
            row[f'{name}_true'] = float(theta4[i])
            row[f'{name}_mean'] = float(means[i])
            row[f'{name}_lo90'] = float(lo[i])
            row[f'{name}_hi90'] = float(hi[i])
        rows.append(row)
track_df = pd.DataFrame(rows)
track_df.to_csv(RESULTS / '27b_sb3_sequential_tracking.csv', index=False)
track_df.head()

## 3. Generate Sequential Observation Windows and Run S-B3 Inference

This stage mirrors notebook 27's window-generation plus SBI-inference core, but under the reduced
4-D scope. It simulates the synthetic sequence, summarizes each window, and applies the selected
`S-B3` posterior to recover the endogenous degradation trajectory.

## 2. Define 30-Day Reduced-Scope Degradation Profile

Instead of the original 5-D trajectory including a feed dimension, this notebook tracks only
reactor-side and column-side endogenous degradation paths under the reduced-scope `S-B3` claim.
The following code block both defines those profiles and runs the reduced-scope sequential SBI
pass over the resulting observation windows.

In [ ]:
# Reduced-scope reference baseline: naive persistence / nominal comparison
baseline_df = track_df.copy()
for pname in SB3_PARAM_NAMES:
    baseline_df[f'{pname}_baseline'] = baseline_df.groupby('track')[f'{pname}_true'].shift(1).fillna(1.0)
baseline_df.head()

## 4. Reduced-Scope Reference Baseline

Notebook 27 compared against an EKF baseline. The reduced-scope notebook keeps the same structural
slot with a lighter baseline: a nominal/persistence reference showing whether the posterior improves
on simply carrying forward the previous window or assuming healthy nominal operation.

## 5. 30-Day Tracking Figure

The main reduced-scope tracking figure compares the true degradation path against posterior means and
90% intervals, together with the simple baseline.

## 6. Metrics Table

As in notebook 27, the notebook closes with compact tracking-quality metrics rather than only plots.

## Final acceptance criteria

This notebook passes if:
1. the posterior tracks both synthetic degradation paths without NaN/Inf outputs,
2. the mean 90% interval hit rate across the four parameters is at least 0.70,
3. the SBI posterior improves on the simple baseline for at least two of the four parameters,
4. the predicted class sequence is not degenerate across all windows.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for ax, pname in zip(axes.ravel(), SB3_PARAM_NAMES):
    for track_name, g in track_df.groupby('track'):
        ax.plot(g['step'], g[f'{pname}_true'], label=f'{track_name} true')
        ax.plot(g['step'], g[f'{pname}_mean'], '--', label=f'{track_name} SBI mean')
        ax.plot(g['step'], baseline_df.loc[g.index, f'{pname}_baseline'], ':', label=f'{track_name} baseline')
        ax.fill_between(g['step'], g[f'{pname}_lo90'], g[f'{pname}_hi90'], alpha=0.2)
    ax.set_title(pname)
    ax.set_xlabel('window index')
axes[0, 0].legend(fontsize=7)
plt.savefig(FIGURES / '27b_sb3_tracking_overview.png', dpi=200, bbox_inches='tight')
plt.show()

metrics_rows = []
for pname in SB3_PARAM_NAMES:
    mae = float(np.mean(np.abs(track_df[f'{pname}_mean'] - track_df[f'{pname}_true'])))
    baseline_mae = float(np.mean(np.abs(baseline_df[f'{pname}_baseline'] - track_df[f'{pname}_true'])))
    hit90 = float(((track_df[f'{pname}_lo90'] <= track_df[f'{pname}_true']) & (track_df[f'{pname}_true'] <= track_df[f'{pname}_hi90'])).mean())
    metrics_rows.append({'parameter': pname, 'sbi_mae': mae, 'baseline_mae': baseline_mae, 'hit90': hit90})
metrics_df = pd.DataFrame(metrics_rows)
metrics_df

## Final acceptance criteria

This notebook passes if:
1. the posterior tracks both synthetic degradation paths without NaN/Inf outputs,
2. the mean 90% interval hit rate across the four parameters is at least 0.70,
3. the SBI posterior improves on the simple baseline for at least two of the four parameters,
4. the predicted class sequence is not degenerate across all windows.

In [ ]:
hit_rates = {row['parameter']: row['hit90'] for _, row in metrics_df.iterrows()}
criteria = [
    ('all tracking outputs finite', bool(np.isfinite(track_df.select_dtypes(include=[float, int]).to_numpy()).all()), 'finite check'),
    ('mean 90% interval hit rate >= 0.70', float(np.mean(list(hit_rates.values()))) >= 0.70, hit_rates),
    ('SBI beats baseline for at least two parameters', int((metrics_df['sbi_mae'] < metrics_df['baseline_mae']).sum()) >= 2, metrics_df[['parameter', 'sbi_mae', 'baseline_mae']].to_dict('records')),
    ('predicted classes not globally degenerate', track_df['pred_class'].nunique() > 1, f"n_unique={track_df['pred_class'].nunique()}"),
]
for name, passed, value in criteria:
    print(f"{'PASS' if passed else 'FAIL'} | {name}: {value}")